# 01_build_kg_ollama_graph_transformer.ipynb

This version swaps the manual per-chunk extraction for **LangChain's LLMGraphTransformer** (like in Tomaz Bratanic's deep-dive).

Flow:

1) Load `data/chunks.jsonl` as LangChain `Document`s (each chunk becomes a source doc)
2) `LLMGraphTransformer` extracts nodes + relationships
3) Import into Neo4j via `Neo4jGraph.add_graph_documents(include_source=True, baseEntityLabel=True)`
4) Convert imported source `:Document` nodes into `:Chunk` nodes and attach Ollama embeddings + a vector index


In [1]:
# %pip install -q python-dotenv tqdm neo4j langchain langchain-community langchain-experimental langchain-ollama pydantic

import os, json
from pathlib import Path
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")              # Aura: neo4j+s://<id>.databases.neo4j.io
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_LLM_MODEL = os.getenv("OLLAMA_LLM_MODEL", "qwen3")
OLLAMA_EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "all-minilm:l12-v2")

# If qwen3/tools aren't supported in your wrapper, set this true to force prompt-mode:
IGNORE_TOOL_USAGE = os.getenv("IGNORE_TOOL_USAGE", "false").lower() == "true"

assert NEO4J_URI and NEO4J_USERNAME and NEO4J_PASSWORD, "Missing Neo4j env vars"


## Connect to Neo4j with `Neo4jGraph`

In [2]:
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    refresh_schema=False,
)

graph.query("RETURN 1 AS ok")


C:\Users\tarmo\AppData\Local\Temp\ipykernel_21544\3813301133.py:3: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


[{'ok': 1}]

## Create constraints/indexes (safe to re-run)

In [3]:
schema = [
    "CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (c:Chunk) REQUIRE c.id IS UNIQUE",
    # baseEntityLabel=True makes extracted nodes have :__Entity__ + their type label
    "CREATE CONSTRAINT entity_id IF NOT EXISTS FOR (e:__Entity__) REQUIRE e.id IS UNIQUE",
    "CREATE INDEX entity_id IF NOT EXISTS FOR (e:__Entity__) ON (e.id)",
]
for q in schema:
    graph.query(q)
print("✅ schema ready")


✅ schema ready


## Load chunks.jsonl into LangChain `Document`s

In [4]:
from langchain_core.documents import Document

CHUNKS_PATH = Path("data/chunks.jsonl")
assert CHUNKS_PATH.exists(), f"Missing {CHUNKS_PATH}. Run the chunking step first."

docs = []
with CHUNKS_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        docs.append(
            Document(
                page_content=obj["text"],
                metadata={
                    "id": obj["chunk_id"],          # used as source doc id in Neo4j
                    "doc_id": obj["doc_id"],
                    **obj.get("metadata", {}),
                },
            )
        )

len(docs), docs[0].metadata


(153,
 {'id': "Alice's Adventures In Wonderland.pdf#000002",
  'doc_id': "Alice's Adventures In Wonderland.pdf",
  'source': "pdf/Alice's Adventures In Wonderland.pdf",
  'page_start': None,
  'page_end': None})

## LLMGraphTransformer extraction

In [5]:
import os, time
from pathlib import Path

try:
    from langchain_ollama import ChatOllama
except Exception:
    from langchain_community.chat_models import ChatOllama

from langchain_experimental.graph_transformers import LLMGraphTransformer

# ---- knobs (env overrideable) ----
REQUEST_TIMEOUT = int(os.getenv("OLLAMA_REQUEST_TIMEOUT", "120"))
MAX_RETRIES = int(os.getenv("LLM_MAX_RETRIES", "3"))
CHECKPOINT_EVERY = int(os.getenv("CHECKPOINT_EVERY", "5"))  # import every N docs
SLEEP_BASE = float(os.getenv("LLM_RETRY_SLEEP_BASE", "2.0"))
OLLAMA_LLM_MODEL=os.getenv("OLLAMA_LLM_MODEL")
# resume files
DONE_PATH = Path("data/done_chunks.txt")
DONE_PATH.parent.mkdir(parents=True, exist_ok=True)

done = set()
if DONE_PATH.exists():
    done = set(x.strip() for x in DONE_PATH.read_text(encoding="utf-8").splitlines() if x.strip())

def mark_done(chunk_id: str):
    with DONE_PATH.open("a", encoding="utf-8") as f:
        f.write(chunk_id + "\n")

# ---- build LLM + transformer ----
# (some versions may ignore request_timeout; it's still worth setting)
llm = ChatOllama(
    model=OLLAMA_LLM_MODEL,
    base_url=OLLAMA_BASE_URL,
    request_timeout=REQUEST_TIMEOUT,
    reasoning=False
)

allowed_nodes = ["Person", "Place", "Organization", "Event", "Object", "Concept"]
allowed_relationships = ["TALKS WITH", "LOCATED_IN", "INTERACTS_WITH"]

transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=allowed_nodes,
    allowed_relationships=allowed_relationships,
    ignore_tool_usage=IGNORE_TOOL_USAGE,
)

# ---- extract + import incrementally ----
graph_documents = []
pending = 0
seen = 0

for d in tqdm(docs, desc="LLMGraphTransformer"):
    chunk_id = d.metadata.get("id") or d.metadata.get("chunk_id")
    if chunk_id and chunk_id in done:
        continue

    ok = False
    last_err = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            gds = transformer.convert_to_graph_documents([d])
            graph_documents.extend(gds)
            ok = True
            break
        except Exception as e:
            last_err = e
            time.sleep(SLEEP_BASE * attempt)

    if not ok:
        print(f"failed on chunk {chunk_id}: {last_err}")
        # skip it so the run continues (remove next line if you want to retry on next run)
        if chunk_id:
            mark_done(chunk_id)
        continue

    if chunk_id:
        mark_done(chunk_id)

    pending += 1
    seen += 1

    if pending >= CHECKPOINT_EVERY:
        graph.add_graph_documents(graph_documents, include_source=True, baseEntityLabel=True)
        graph_documents = []
        pending = 0

# final flush
if graph_documents:
    graph.add_graph_documents(graph_documents, include_source=True, baseEntityLabel=True)

print(f"✅ imported (processed {seen} new chunks; skipped {len(done)} already-done)")


LLMGraphTransformer:   0%|          | 0/153 [00:00<?, ?it/s]

✅ imported (processed 47 new chunks; skipped 106 already-done)


## Label imported sources as `:Chunk` + attach Ollama embeddings

In [6]:
try:
    from langchain_ollama import OllamaEmbeddings
except Exception:
    from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)

# Add :Chunk label to imported source nodes.
# Depending on LangChain version, the source node label is usually :Document.
graph.query("""
MATCH (d:Document)
WHERE d.id IS NOT NULL
SET d:Chunk
""")

# Try to fill doc_id from metadata if present
graph.query("""
MATCH (c:Chunk)
WHERE c.metadata IS NOT NULL AND c.doc_id IS NULL AND c.metadata.doc_id IS NOT NULL
SET c.doc_id = c.metadata.doc_id
""")

def ensure_vector_index(dim: int):
    graph.query(f"""
    CREATE VECTOR INDEX chunk_embedding IF NOT EXISTS
    FOR (c:Chunk) ON (c.embedding)
    OPTIONS {{ indexConfig: {{
      `vector.dimensions`: {dim},
      `vector.similarity_function`: 'cosine'
    }} }}
    """)

dim = len(embeddings.embed_query("hello"))
ensure_vector_index(dim)

# Embed chunks missing embeddings
rows = graph.query("MATCH (c:Chunk) WHERE c.embedding IS NULL RETURN c.id AS id, c.text AS text LIMIT 100000")
print("Chunks missing embedding:", len(rows))

BATCH = 25
buf = []
for r in tqdm(rows, desc="Embedding"):
    buf.append({"id": r["id"], "embedding": embeddings.embed_query(r["text"] or "")})
    if len(buf) >= BATCH:
        graph.query("""
        UNWIND $rows AS row
        MATCH (c:Chunk {id: row.id})
        SET c.embedding = row.embedding
        """, {"rows": buf})
        buf = []
if buf:
    graph.query("""
    UNWIND $rows AS row
    MATCH (c:Chunk {id: row.id})
    SET c.embedding = row.embedding
    """, {"rows": buf})

print("✅ embeddings done")


Chunks missing embedding: 47


Embedding:   0%|          | 0/47 [00:00<?, ?it/s]

✅ embeddings done


## Quick sanity checks

In [7]:
graph.query('MATCH (c:Chunk) RETURN count(c) AS chunks')[0]

{'chunks': 122}

In [8]:
graph.query('MATCH (e:__Entity__) RETURN count(e) AS entities')[0]

{'entities': 169}

In [9]:
graph.query('MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n ORDER BY n DESC LIMIT 10')

[{'rel': 'MENTIONS', 'n': 444},
 {'rel': 'INTERACTS_WITH', 'n': 3},
 {'rel': 'LOCATED_IN', 'n': 1}]